# square_map: chess squares <-> world coordinates

Library notebook for `gambit_arm`. Converts a square name (`"e4"`) to the world point the
gripper must reach, and back (nearest square to an (x, y) point, e.g. from the camera).
Everything comes from `gambit_bringup/config/board.yaml`, so measuring the real board and
editing that file is all it takes to move from Gazebo to the real table.

Milestone: *Hardware and execution* (board position registered; pick-and-place needs targets).

**TODO**
- [ ] load board.yaml (searching up from this folder, works in Colab after `%cd`) (given: kept working so the other notebooks can run)
- [ ] square -> world (x, y, z) with the docs/gazebo.md formula, incl. `white_near_arm: false`
- [ ] world (x, y) -> nearest square, with an outside-the-board check
- [ ] graveyard slots (row along +x, same spacing as the Gazebo world)
- [ ] piece heights and grasp heights from `pieces.height` and `arm.grasp_height_frac`
- [ ] after registering the real board (ArUco corners), replace `board.center` with the
      measured value and add the measured yaw if the board is rotated (formula below assumes 0)
- [ ] decide how many graveyard slots fit within reach (a second row?) once IK is validated
- [ ] write the self-checks at the bottom as asserts

In [ ]:
from pathlib import Path
import math
import numpy as np
import yaml

## Finding the repo and loading board.yaml

Notebooks run with their own folder as the working directory (that is how `%run` paths
resolve), so walk upward until we find `gambit_bringup/config`.

These two helpers are left implemented on purpose: every other gambit_arm notebook needs
`BOARD` / `SAFETY` at import time, and config loading is plumbing, not robotics.

In [ ]:
def find_repo_root(start=None):
    """Return the repo root (the folder containing gambit_bringup/config)."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "gambit_bringup" / "config").is_dir():
            return cand
    raise FileNotFoundError("could not find gambit_bringup/config above " + str(p))

REPO_ROOT = find_repo_root()
CONFIG_DIR = REPO_ROOT / "gambit_bringup" / "config"

def load_yaml(name):
    with open(CONFIG_DIR / name) as f:
        return yaml.safe_load(f)

BOARD = load_yaml("board.yaml")
BOARD["board"]

## Square -> world

From docs/gazebo.md: with file 0 = a and rank 0 = rank 1,

$$x = c_x + (\text{rank} - 3.5)\,s, \qquad y = c_y + (3.5 - \text{file})\,s, \qquad z = \text{thickness}$$

Rank grows away from the arm (+x); file a is on the arm's left (+y). If the board is turned
around (`white_near_arm: false`) both indices flip. `z` is the board top; the grasp height
is added on top of it per piece.

In [ ]:
FILES = "abcdefgh"

def parse_square(name) -> tuple:
    """'e4' -> (file=4, rank=3). Also accepts a python-chess index 0..63.

    Raises ValueError on anything that is not a valid square.
    """
    # Design:
    # - Two input types: an int index (python-chess numbering: a1 = 0, b1 = 1, ..., h8 = 63,
    #   so file = index mod 8, rank = index div 8) or a 2-char string (letter in FILES, digit 1-8).
    # - Accept numpy integers too (indices often come out of arrays).
    # - Validate strictly and raise ValueError with the offending value: silent garbage here
    #   turns into the arm reaching for the wrong place.
    # TODO: implement
    raise NotImplementedError

def square_name(file: int, rank: int) -> str:
    """(file, rank) 0-based -> 'e4'. Inverse of parse_square for strings."""
    # Design: one line, letter from FILES + rank number (1-based).
    # TODO: implement
    raise NotImplementedError

def square_to_world(name, cfg=BOARD) -> np.ndarray:
    """Square center on the board top, world frame, meters -> np.array([x, y, z])."""
    # Design:
    # - parse_square -> (file, rank); read square size, center and thickness from cfg["board"].
    # - If white_near_arm is false the board is rotated 180 deg: flip both indices (7 - i)
    #   before applying the formula above.
    # - Apply the formula in the markdown cell; z is the board top (thickness).
    # Check against docs/gazebo.md: e4 is at (0.2025, -0.0125, 0.006).
    # TODO: implement
    raise NotImplementedError

## World -> nearest square

Invert the formula, round to the nearest index, and reject points outside the 8x8 playing
area (more than half a square beyond the outer centers).

In [ ]:
def world_to_square(x: float, y: float, cfg=BOARD):
    """Nearest square name to world (x, y), or None if outside the playing area."""
    # Design:
    # - Solve the square_to_world formula for continuous (rank, file) indices.
    # - Outside check BEFORE rounding: valid continuous indices lie in [-0.5, 7.5).
    # - Round to the nearest integer, clamp to 0..7 (edge rounding), then undo the
    #   white_near_arm flip and build the name.
    # Decision for you: should points on the frame (border) map to None or to the edge square?
    # TODO: implement
    raise NotImplementedError

def offset_from_center(x: float, y: float, cfg=BOARD):
    """(square, dx, dy): nearest square and the offset of (x, y) from its center (m).
    Used to check that a placed piece is centered within tolerance."""
    # Design: world_to_square, then subtract square_to_world of that square. Return
    # (None, nan, nan) when off the board so callers can test with math.isnan.
    # TODO: implement
    raise NotImplementedError

## Graveyard and piece heights

Captured pieces go to a row along +x starting at `pieces.graveyard`, on the table (z = 0).
The spacing (0.9 square) matches `GRAVEYARD_SPACING` in gambit_sim/notebooks/world_generator.ipynb
so the Gazebo world's spare queens sit exactly on slots 0..3.

Grasp height = board top (or table) + `grasp_height_frac` * piece height.

In [ ]:
GRAVEYARD_SPACING_FRAC = 0.9

def graveyard_slot(index: int, cfg=BOARD) -> np.ndarray:
    """World point of graveyard slot `index` (0, 1, ...) on the table top."""
    # Design: start at pieces.graveyard (x, y), step along +x by GRAVEYARD_SPACING_FRAC squares
    # per slot, z = 0 (table, not board top). Spare queens in chess_world.sdf sit on slots 0..3.
    # TODO: implement
    raise NotImplementedError

def piece_height(kind: str, cfg=BOARD) -> float:
    """Piece height in m for kind in 'PNBRQK' (either case)."""
    # Design: pieces.height stores fractions of the square edge; multiply by board.square.
    # TODO: implement
    raise NotImplementedError

def grasp_point(target, kind: str, cfg=BOARD) -> np.ndarray:
    """Where the gripper tip should be to grasp piece `kind` standing at `target`
    (a square name or a world point such as a graveyard slot)."""
    # Design:
    # - Base point: square_to_world if target is a square (str or int), else the given point.
    # - Add arm.grasp_height_frac * piece_height(kind) in z: grasping around the middle of the
    #   piece is more stable than at the top, and keeps the fingers off the board.
    # TODO: implement
    raise NotImplementedError

def hover_point(target, cfg=BOARD) -> np.ndarray:
    """Point `arm.hover_height` above the board top (or table) at the target."""
    # Design: same base point logic as grasp_point, plus arm.hover_height in z.
    # TODO: implement
    raise NotImplementedError

# ALL_SQUARES: the 64 names in python-chess order (a1, b1, ..., h8).
# TODO: build it with square_name once that works:
# ALL_SQUARES = [square_name(f, r) for r in range(8) for f in range(8)]
ALL_SQUARES = [f"{f}{r}" for r in range(1, 9) for f in FILES]

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - docs/gazebo.md example: square_to_world("e4") == (0.2025, -0.0125, 0.006); e2 -> x 0.1525
# - corners: a1 -> (0.1275, 0.0875), h8 -> (0.3025, -0.0875)
# - python-chess index 28 gives the same point as "e4"; bad names ("i9", "e", 64) raise ValueError
# - round trip: for every square, world_to_square(x + 0.01, y - 0.01) returns that square
# - world_to_square(0, 0) is None (off the board)
# - offset_from_center(0.2045, -0.0125) == ("e4", 0.002, 0.0)
# - white_near_arm: false -> h8 lands where a1 was, and world_to_square inverts it
# - graveyard slots 0..3 at x = 0.17, 0.1925, 0.215, 0.2375 (Gazebo spare queens)
# - grasp_point("e4", "P") z == 0.006 + grasp_height_frac * 0.9 * 0.025; hover_point("e4") z == 0.076
# Example shape of a test:
# assert np.allclose(square_to_world("e4"), [0.2025, -0.0125, 0.006])
print("Self-check: not written yet (see plan above)")
print("square_map OK")